# SurvFace — 01. Aligned crop materialization

SurvFace training과 공식 gallery/mated/unmated 전체를 RGB uint8 NHWC 112×112 crop으로 정렬합니다. 실패를 center crop으로 대체하지 않습니다.

이 노트북은 한 단계만 실행하는 thin runbook입니다. 계산 구현은 `research/experiments/step4_workflow.py`에 있습니다.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
EXECUTION = CONFIG["execution"]
MODEL_PROFILE = str(EXECUTION["model_profile"])
MODE = str(EXECUTION["mode"])
DATA_FRACTION = float(EXECUTION["data_fraction"])
EXECUTE_STAGE = bool(EXECUTION["execute_stage"])
WRITE_OUTPUTS = bool(EXECUTION["write_outputs"])
OVERWRITE = bool(EXECUTION["overwrite"])
DATASET_ID = "survface"

if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if MODE == "real" and DATA_FRACTION != 1.0:
    raise ValueError("real 모드는 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")
if EXECUTE_STAGE and not WRITE_OUTPUTS:
    raise ValueError("정식 단계 실행은 WRITE_OUTPUTS=True여야 합니다.")

from research.experiments import materialize_step4_aligned_crops
from research.runtime import ProgressReporter

PROGRESS = ProgressReporter(
    "SurvFace aligned crops",
    heartbeat_seconds=None,
    milestone_percent=10,
)


In [2]:
if EXECUTE_STAGE:
    result = materialize_step4_aligned_crops(
        CONFIG_PATH,
        project_root=PROJECT_ROOT,
        dataset_id=DATASET_ID,
        execution_acknowledged=True,
        progress=PROGRESS.callback(key_prefix=f"{DATASET_ID}:"),
    )
else:
    result = {
        "dataset_id": DATASET_ID,
        "status": "not_executed",
        "reason": "CONFIG execution gates are closed",
    }

result


[23:30:26] SurvFace aligned crops | aligned crop materialization | elapsed=1m 23s | progress=10% processed=46335 total=463341 rate=559.79/s eta=12m 25s aligned=46335 failed=0
[23:31:42] SurvFace aligned crops | aligned crop materialization | elapsed=2m 39s | progress=20% processed=92669 total=463341 rate=583.72/s eta=10m 35s aligned=92669 failed=0
[23:32:56] SurvFace aligned crops | aligned crop materialization | elapsed=3m 53s | progress=30% processed=139003 total=463341 rate=597.85/s eta=9m 03s aligned=139003 failed=0
[23:34:10] SurvFace aligned crops | aligned crop materialization | elapsed=5m 07s | progress=40% processed=185337 total=463341 rate=603.74/s eta=7m 40s aligned=185337 failed=0
[23:35:26] SurvFace aligned crops | aligned crop materialization | elapsed=6m 22s | progress=50% processed=231671 total=463341 rate=605.92/s eta=6m 22s aligned=231671 failed=0
[23:36:40] SurvFace aligned crops | aligned crop materialization | elapsed=7m 37s | progress=60% processed=278005 total=46

{'dataset_id': 'survface',
 'aligned_bundle_dir': 'C:\\ronbun\\data\\interim\\step4\\survface\\aligned_112',
 'complete': True}

## 다음 단계

다음은 `02_landmark_region_materialization.ipynb`입니다.

커널을 재시작한 뒤 다음 노트북을 위에서 아래로 실행합니다.